# 02 - Multi-Interface Edge Delamination

This notebook follows the Multi-Interface Edge Delamination documentation. It shows edge delamination on its own: first on a single interface, then across two interfaces with hierarchical attribution.

This is an edge-only workflow -- no crack detection and no diffuse delamination are involved.

## Path handling (hide me)

In [1]:
from pathlib import Path

from deladect.detection import DelaminationDetector
from deladect.specimen import Specimen


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root (missing {marker!r}) above {Path.cwd()}")


repo_root = find_repo_root()
data_root = repo_root / "example_images" / "sample-3"

## 1. Building the specimen

Multi-interface delamination needs at least two interfaces, so this specimen has three plies (`[0, 90, 0]`) and two interfaces: `i0` between the first two plies, and `i1` between the last two. `i0` is the primary interface; `i1` is the deeper interface that can be attributed damage once evidence of delamination beneath `i0` persists.

Crack detection isn't needed for this workflow, so no ply-level crack parameters are set beyond the defaults.

In [2]:
specimen = Specimen(
    name="02-multi-interface-edge",
    scale_px_mm=41.03328366,
    path_full=str(data_root),
    sorting_key="_sc",
    image_types=["png"],
    results_root=str(repo_root / "results"),
    avg_crack_width_px=8.0,
)

for index, orientation in enumerate((0.0, 90.0, 0.0)):
    specimen.add_ply(
        name=f"ply_{index}",
        orientation_deg=orientation,
        avg_crack_width_px=8.0,
        min_crack_length_px=20.0,
    )

for index in range(2):
    specimen.add_interface(name=f"i{index}", upper_ply=index, lower_ply=index + 1)

detector = DelaminationDetector(specimen, specimen.interfaces[0], save_preprocess_outputs=True)

## 2. Standalone edge delamination (single interface)

`detector.edge.detect_primary` runs entirely on its own -- no crack catalogue, no diffuse pipeline, and no second interface required. This is the same edge algorithm used inside `detect_both_delaminations`, just called directly.

In [3]:
from deladect.io.delamination import save_mask_bundle

primary_only = detector.edge.detect_primary(
    save_overlays=True,
    overlay_dirname="edge_only",
    params={"window_edge": (1, 60), "seed_ratio": 0.01},
)
primary_only_masks_path = save_mask_bundle(
    primary_only["masks"],
    specimen.results_dir("edge_only", "edge", "masks") / "primary.npz",
)

single_interface_overlays = specimen.results_dir("edge_only", "edge", "overlays")
single_interface_overlays

PosixPath('/mnt/c/Users/p2321038/Documents/GitHub/Deladect/results/02-multi-interface-edge/edge_only/edge/overlays')

## 3. Multi-interface delamination (i0 + i1)

`detect_edge_multi` accepts two separate preprocessing caches: a *static*-reference cache drives the primary accumulation at `i0`, while a *rolling-median* cache drives the deeper-interface check at `i1` -- it needs to stay sensitive to change happening inside a region that's already flagged as damaged, which a static reference would no longer highlight.

In [4]:
primary_cache = detector.preprocess_stack_to_disk(
    specimen.image_stack_full,
    key="primary_static",
    reference_mode="static",
)["cache_paths"]

secondary_cache = detector.preprocess_stack_to_disk(
    specimen.image_stack_full,
    key="secondary_rolling",
    reference_mode="rolling_median",
    reference_window=7,
    reference_skip=2,
)["cache_paths"]

multi_result = detector.edge.detect_edge_multi(
    interfaces=specimen.interfaces,
    processed_cache_paths=primary_cache,
    secondary_cache_paths=secondary_cache,
    save_masks=True,
    save_overlays=True,
    primary_params={"window_edge": (1, 60), "seed_ratio": 0.01},
    secondary_params={"secondary_similarity_threshold": 0.6},
)

multi_result["paths"]

{'inclusive_masks': {'i0': '/mnt/c/Users/p2321038/Documents/GitHub/Deladect/results/02-multi-interface-edge/delamination/edge_multi/masks/i0_inclusive.npz',
  'i1': '/mnt/c/Users/p2321038/Documents/GitHub/Deladect/results/02-multi-interface-edge/delamination/edge_multi/masks/i1_inclusive.npz'},
 'exclusive_masks': {'i0': '/mnt/c/Users/p2321038/Documents/GitHub/Deladect/results/02-multi-interface-edge/delamination/edge_multi/masks/i0_exclusive.npz',
  'i1': '/mnt/c/Users/p2321038/Documents/GitHub/Deladect/results/02-multi-interface-edge/delamination/edge_multi/masks/i1_exclusive.npz'},
 'overlays': '/mnt/c/Users/p2321038/Documents/GitHub/Deladect/results/02-multi-interface-edge/delamination/edge_multi/overlays'}

The saved overlay shows both interfaces classified with distinct colors in a single frame, so the attributed `i1` region is directly visible against the primary `i0` accumulation.

For the reasoning behind the two preprocessing caches, the attribution parameters (`secondary_similarity_threshold`, `min_primary_frac_for_secondary`, `secondary_start_frame`), and how the attribution logic itself works, see the *Multi-Interface Delamination* page in the User Guide.